# Clase 090 — Stacking (stacked generalization)

Combinamos modelos heterogéneos entrenando un **meta-modelo (blender)** sobre las
predicciones **out-of-fold** de los base learners, con `StackingClassifier` de sklearn
(que hace el CV interno y evita leakage).

Requiere: `numpy`, `scikit-learn`, `matplotlib`.

## 1. Dataset `make_classification`

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import (RandomForestClassifier, StackingClassifier,
                              VotingClassifier)
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression

np.random.seed(42)

X, y = make_classification(
    n_samples=2000, n_features=20, n_informative=10, n_redundant=4,
    random_state=42)
print('X', X.shape, 'y', y.shape)

## 2. Base learners heterogéneos por separado (CV 5-fold)

In [ ]:
base = [
    ('rf',  RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=1)),
    ('svc', SVC(probability=True, random_state=42)),
    ('knn', KNeighborsClassifier(n_neighbors=15)),
]

acc_base = {}
for name, clf in base:
    scores = cross_val_score(clf, X, y, cv=5, n_jobs=1)
    acc_base[name] = scores
    print(f'{name:4s} CV acc {scores.mean():.4f} +/- {scores.std():.4f}')

best_base = max(s.mean() for s in acc_base.values())
print(f'\nmejor base learner: {best_base:.4f}')

## 3. StackingClassifier con blender logístico

El blender ve las predicciones **out-of-fold** de los base (CV interna) — sin leakage.

In [ ]:
stack = StackingClassifier(
    estimators=base,
    final_estimator=LogisticRegression(random_state=42),
    cv=5, n_jobs=1)
scores_stack = cross_val_score(stack, X, y, cv=5, n_jobs=1)
print(f'Stacking CV acc {scores_stack.mean():.4f} +/- {scores_stack.std():.4f}')
assert scores_stack.mean() >= best_base - 0.01, 'stacking debería igualar al mejor base'
print('assert OK: el stacking iguala o supera al mejor base learner')

## 4. `passthrough=True`: el blender ve también las features originales

In [ ]:
stack_pt = StackingClassifier(
    estimators=base,
    final_estimator=LogisticRegression(max_iter=1000, random_state=42),
    cv=5, passthrough=True, n_jobs=1)
scores_pt = cross_val_score(stack_pt, X, y, cv=5, n_jobs=1)
print(f'Stacking passthrough CV acc {scores_pt.mean():.4f} +/- {scores_pt.std():.4f}')

## 5. Stacking vs Voting (soft)

In [ ]:
voting = VotingClassifier(estimators=base, voting='soft', n_jobs=1)
scores_vote = cross_val_score(voting, X, y, cv=5, n_jobs=1)
print(f'Voting (soft) CV acc {scores_vote.mean():.4f} +/- {scores_vote.std():.4f}')
print(f'Stacking      CV acc {scores_stack.mean():.4f} +/- {scores_stack.std():.4f}')

## 6. Comparativa final

In [ ]:
import pandas as pd
filas = []
for name, s in acc_base.items():
    filas.append((name, s.mean(), s.std()))
filas.append(('stacking', scores_stack.mean(), scores_stack.std()))
filas.append(('stack+passthrough', scores_pt.mean(), scores_pt.std()))
filas.append(('voting_soft', scores_vote.mean(), scores_vote.std()))
tabla = pd.DataFrame(filas, columns=['modelo', 'acc_mean', 'acc_std']).round(4)
print(tabla.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(tabla.modelo, tabla.acc_mean, yerr=tabla.acc_std,
       color=['#aaa', '#aaa', '#aaa', '#3a7', '#37a', '#a73'])
ax.axhline(best_base, ls='--', color='k', lw=0.8, label=f'mejor base = {best_base:.3f}')
ax.set_ylim(min(tabla.acc_mean) - 0.03, max(tabla.acc_mean) + 0.03)
ax.set_ylabel('CV accuracy (5-fold)')
ax.set_title('Stacking aprende a combinar; voting promedia con pesos fijos')
ax.legend()
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## Ejercicios

1. Variá `cv` en {3, 5, 10} dentro del stacking y medí accuracy vs tiempo de
   entrenamiento (trade-off típico).
2. Usá un blender complejo (`RandomForestClassifier`) y observá cómo tiende a
   overfittear las predicciones OOF.
3. Reemplazá un base learner por otro árbol para reducir la diversidad; ¿cae la ganancia
   del stacking?
4. Medí el costo computacional: stacking entrena `M base x K folds + blender`.

## Conclusiones

- Stacking entrena un **meta-modelo** sobre predicciones out-of-fold: aprende los pesos
  (y combinaciones) en vez de fijarlos como el voting.
- El CV interno de `StackingClassifier` evita el leakage automáticamente.
- El blender conviene **simple y regularizado** (logística/Ridge) para no overfittear.
- Suele ganar 1-3 pp al mejor base, a costa de 5-10x más cómputo: úsalo cuando el upside
  lo justifique.